# TradeSense · Component 03 · notebook 37: features v3-lite for XGBoost (CPU, ~20–40 min)

Can richer **multi-scale features** (1 h and 4 h context, volume / trade-size ratios, large-trade share, flow accumulation, run length) lift the weak families, with the same XGBoost, the same rows and the same rule as v2? Everything below is pre-registered in **amendment A4** (project doc `protocol_b2_amendment_A4_features_v3lite.md`, written before any result). TEST-B / TEST-3 are never loaded; CAL-3 is evaluated once for reporting and never used for selection.

What it does: unit tests (feature causality, row masks) → builds set C (53 v2 features + ~46 multi-scale features) → trains XGBoost (3 seeds, resumable, configuration and normalisation as declared for XGBoost v2) → compares with v2 on VAL-3 per family (T1c, 2 episodes/day, paired bootstrap) → applies the acceptance rule → reports CAL-3 once → writes `xgb_v3_decisions.json` (and `declared_final_set_v2.json` only if accepted).

## Amendment A4 (text)
# Protocol B2 – Amendment A4 – 2026-10-14 – features v3-lite for XGBoost (candidate, notebook 37)
Written BEFORE any v3-lite result exists. Notebook 31 (CAL measurement and the declared final set v1) has been written; TEST-B / TEST-3 are untouched. CAL-3 results of v3-lite are reported once and never used for selection.

## Why
Notebook 30 and the A1/A2 tables show weak families (abnormal_volume_burst ≤ 0.24 even for the union, trade_flow_imbalance and wash-like when unseen). The v2 XGBoost features look back at most 15 bars; volume bursts, slow accumulation and flow imbalance can need longer context (1 h, 4 h) and raw-quantity ratios (average trade size, large-trade share, volume and trade-count burst ratios, buy/sell run length).

## What is new (all causal, all computable on the injected series because Layer 3 v3 rewrites volume, trade count and large-trade count)
Set C = the 53 v2 features (set A) + about 46 multi-scale features: rolling mean, max |z|, std and deviation from the previous 240 bars for each of the six normalised features (30); flow accumulation (sum of trade_flow_imbalance over 15/60/240 bars, sum of log_returns over 60/240) (5); log-ratio of the last 5 bars to the previous 240 bars for volume, trade count and average trade size, and of the current bar to the previous 60 bars (6); large-trade share (deviation and 15-bar mean) (2); number of extreme features (mean over 60, max over 15) (2); signed run length of one-sided flow (1). Same normalisation as declared for XGBoost, same hyper-parameters, same row definition, same configuration (feature scope and depth) as the declared XGBoost v2; seeds 11/22/33 averaged. Trade-level features (exact same-size repeats, size distribution) need trade-level injection and are NOT part of v3-lite (listed as a limitation and a future step).

## Acceptance rule (VAL-3 only; T1c; budget 2 episodes/day; mean of BTC/ETH; same bars as the v2 comparator)
v3-lite replaces XGBoost v2 in the final set only if ALL hold: (a) macro recall over all 8 families improves by at least 0.03 and the paired event-bootstrap 95% interval of the difference is above 0; (b) no family drops by more than 0.05; (c) the realised genuine DEV episodes/day (T1) lie in [0.5, 2] x budget. Otherwise v2 stays and v3-lite is reported as a negative result.
If accepted, a new file `declared_final_set_v2.json` is written (v1 stays), and notebook 32 and fusion are re-run on it BEFORE TEST is opened.

## Hypothesis H-V3 (family level, reported either way)
For each family that notebook 36 labels "feature/model gap", v3-lite improves the VAL-3 recall (T1c, budget 2) by at least 0.05 over v2. Families that notebook 36 labels amplitude-limited are expected NOT to improve; improvements there are reported but not expected.

## Limitations stated in advance
Synthetic positives only; the v3-lite features were designed after seeing the weak families on VAL-3 (a design choice made before measuring v3-lite, but informed by v2 results); one seed averaged over three; no LOFO for v3-lite here.

In [ ]:
import os, json, glob, time, hashlib, warnings, sys, subprocess, datetime
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost']); import xgboost as xgb
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; X3 = OUT/'xgb_v3'; X3.mkdir(parents=True, exist_ok=True)
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; SEEDS = [11, 22, 33]
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 200 if MOCK else 1000
MAX_RUN_MINUTES = float(os.environ.get('MAX_RUN_MINUTES', 150)); T_START = time.time(); NTHREAD = max(1, os.cpu_count() or 1)
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
def out_of_time(): return (time.time() - T_START) / 60 > MAX_RUN_MINUTES
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
for f in ['declared_final_set_v1.json', 'xgb_v2_decisions.json']: assert (OUT/f).exists(), f'{f} missing: run notebooks 29 and 31 first'
A4_TEXT = r"""# Protocol B2 – Amendment A4 – 2026-10-14 – features v3-lite for XGBoost (candidate, notebook 37)
Written BEFORE any v3-lite result exists. Notebook 31 (CAL measurement and the declared final set v1) has been written; TEST-B / TEST-3 are untouched. CAL-3 results of v3-lite are reported once and never used for selection.

## Why
Notebook 30 and the A1/A2 tables show weak families (abnormal_volume_burst ≤ 0.24 even for the union, trade_flow_imbalance and wash-like when unseen). The v2 XGBoost features look back at most 15 bars; volume bursts, slow accumulation and flow imbalance can need longer context (1 h, 4 h) and raw-quantity ratios (average trade size, large-trade share, volume and trade-count burst ratios, buy/sell run length).

## What is new (all causal, all computable on the injected series because Layer 3 v3 rewrites volume, trade count and large-trade count)
Set C = the 53 v2 features (set A) + about 46 multi-scale features: rolling mean, max |z|, std and deviation from the previous 240 bars for each of the six normalised features (30); flow accumulation (sum of trade_flow_imbalance over 15/60/240 bars, sum of log_returns over 60/240) (5); log-ratio of the last 5 bars to the previous 240 bars for volume, trade count and average trade size, and of the current bar to the previous 60 bars (6); large-trade share (deviation and 15-bar mean) (2); number of extreme features (mean over 60, max over 15) (2); signed run length of one-sided flow (1). Same normalisation as declared for XGBoost, same hyper-parameters, same row definition, same configuration (feature scope and depth) as the declared XGBoost v2; seeds 11/22/33 averaged. Trade-level features (exact same-size repeats, size distribution) need trade-level injection and are NOT part of v3-lite (listed as a limitation and a future step).

## Acceptance rule (VAL-3 only; T1c; budget 2 episodes/day; mean of BTC/ETH; same bars as the v2 comparator)
v3-lite replaces XGBoost v2 in the final set only if ALL hold: (a) macro recall over all 8 families improves by at least 0.03 and the paired event-bootstrap 95% interval of the difference is above 0; (b) no family drops by more than 0.05; (c) the realised genuine DEV episodes/day (T1) lie in [0.5, 2] x budget. Otherwise v2 stays and v3-lite is reported as a negative result.
If accepted, a new file `declared_final_set_v2.json` is written (v1 stays), and notebook 32 and fusion are re-run on it BEFORE TEST is opened.

## Hypothesis H-V3 (family level, reported either way)
For each family that notebook 36 labels "feature/model gap", v3-lite improves the VAL-3 recall (T1c, budget 2) by at least 0.05 over v2. Families that notebook 36 labels amplitude-limited are expected NOT to improve; improvements there are reported but not expected.

## Limitations stated in advance
Synthetic positives only; the v3-lite features were designed after seeing the weak families on VAL-3 (a design choice made before measuring v3-lite, but informed by v2 results); one seed averaged over three; no LOFO for v3-lite here.
"""; A4H = hashlib.sha256(A4_TEXT.encode()).hexdigest(); (X3/'amendment_A4.md').write_text(A4_TEXT)
say('protocol B2 intact, sha256 ' + ph[:16] + '; amendment A4 sha256 ' + A4H[:16] + '; xgboost ' + xgb.__version__ + f'; {NTHREAD} threads' + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f; amendment A4 sha256 3986409d2b6aacd8; xgboost 3.4.1; 2 threads


## 1 · Library (same file as notebooks 28-36)

In [ ]:
_old_lib = (OUT/'b2_lib.py').read_text() if (OUT/'b2_lib.py').exists() else None
open("b2_lib.py", "w").write(r'''
"""TradeSense C03 - Protocol B2 shared library: normalisation N0/N1, conditional autoencoder (AE-ctx), burden thresholds T0/T1, recall."""
import numpy as np, pandas as pd, torch, torch.nn as nn, time, json, hashlib
from numpy.lib.stride_tricks import sliding_window_view

FEATS = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
LOGC = [3, 5]; WIN7 = 10080; MINP = 1440; CTX = 15; WARM = 30; GAPM = 5; CLIP = 20.0
BUDGETS = [1, 2, 4]; PRIMARY = 2; QGRID = [0.99, 0.993, 0.995, 0.997, 0.998, 0.999, 0.9993, 0.9995, 0.9997, 0.9998, 0.9999, 0.99995]
FAMS = {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance', 5: 'wash_like_aggregate_proxy',
        6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}


# ---------------------------------------------------------------- normalisation
def to_x(F):
    x = np.asarray(F[FEATS] if hasattr(F, 'columns') else F, dtype='float64').copy()
    for j in LOGC: x[:, j] = np.log1p(x[:, j])
    return x


def fit_n0(F_train):
    """median / (IQR/1.349) on log1p-transformed Layer 1 TRAIN bars (caller passes bars >= 15 only)."""
    x = to_x(F_train); med = np.median(x, axis=0); q75, q25 = np.percentile(x, [75, 25], axis=0); return {'median': med, 'scale': (q75 - q25) / 1.349}


def normalise(F, mode, n0):
    x = to_x(F)
    if mode == 'N0': med, scl = n0['median'][None, :], n0['scale'][None, :]
    else:
        df = pd.DataFrame(x); r = df.rolling(WIN7, min_periods=MINP)
        med = r.median().shift(1).to_numpy(); q75 = r.quantile(0.75).shift(1).to_numpy(); q25 = r.quantile(0.25).shift(1).to_numpy()
        scl = np.maximum((q75 - q25) / 1.349, 0.1 * n0['scale'][None, :])
        bad = np.isnan(med) | np.isnan(scl); med = np.where(bad, n0['median'][None, :], med); scl = np.where(bad, n0['scale'][None, :], scl)
    return np.clip((x - med) / scl, -CLIP, CLIP)


def context(z):
    """causal context: mean and std (ddof 1) of the previous CTX bars; rows < CTX are zero-filled and never used."""
    n = len(z); m = np.zeros_like(z); s = np.zeros_like(z)
    w = sliding_window_view(z, CTX, axis=0)[: n - CTX]            # window k = bars k..k+14 -> bar k+15
    m[CTX:] = w.mean(axis=2); s[CTX:] = w.std(axis=2, ddof=1); return m, s


# ---------------------------------------------------------------- conditional autoencoder
class AECtx(nn.Module):
    def __init__(self, d=6, b=3):
        super().__init__(); self.enc = nn.Sequential(nn.Linear(d + 2 * d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, b))
        self.dec = nn.Sequential(nn.Linear(b + 2 * d, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, d))

    def forward(self, z, cm, cs):
        c = torch.cat([cm, cs], 1); return self.dec(torch.cat([self.enc(torch.cat([z, c], 1)), c], 1))


def train_ae(z, idx, seed, max_epochs=150, patience=10, batch=512, log=None):
    """z normalised genuine series; idx = bar indices allowed for fitting (valid, inside Layer 1 TRAIN). 90% fit / 10% early stop, 15-bar gap."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); cm, cs = context(z)
    idx = np.sort(idx); cut = int(0.9 * len(idx)); fit = idx[:cut]; val = idx[idx > idx[cut] + CTX]
    T = lambda a: torch.tensor(a, dtype=torch.float32)
    Zf, Mf, Sf = T(z[fit]), T(cm[fit]), T(cs[fit]); Zv, Mv, Sv = T(z[val]), T(cm[val]), T(cs[val])
    net = AECtx(); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); best, bad, state = 1e18, 0, None
    for ep in range(max_epochs):
        net.train(); perm = torch.tensor(rng.permutation(len(fit)))
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]; opt.zero_grad(); loss = ((net(Zf[b], Mf[b], Sf[b]) - Zf[b]) ** 2).mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = float(((net(Zv, Mv, Sv) - Zv) ** 2).mean())
        if v < best - 1e-6: best, bad, state = v, 0, {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval(); return net, {'epochs': ep + 1, 'best_val': best, 'fit_rows': int(len(fit)), 'val_rows': int(len(val)), 'fit_min': int(fit.min()), 'fit_max': int(fit.max())}


def ae_errors(net, z):
    cm, cs = context(z); T = lambda a: torch.tensor(a, dtype=torch.float32)
    with torch.no_grad(): rec = net(T(z), T(cm), T(cs)).numpy()
    return (z - rec) ** 2


def p99_scale(err, idx): return np.percentile(err[idx], 99, axis=0) + 1e-12


def ae_columns(err, scale):
    """8 columns: fmax score, fmean score, 6 per-feature errors / TRAIN p99 scale."""
    r = err / scale[None, :]; return np.column_stack([r.max(axis=1), err.mean(axis=1), r])


# ---------------------------------------------------------------- burden: episodes, thresholds, recall
def episodes(alarm, gap=GAPM):
    a = np.flatnonzero(alarm)
    if len(a) == 0: return 0
    return int(1 + np.sum(np.diff(a) > gap))


def eps_per_day(alarm, valid):
    a = alarm & valid; days = valid.sum() / 1440.0
    return episodes(a) / max(days, 1e-9)


def valid_mask(n, lo, hi): m = np.zeros(n, bool); m[max(lo, WARM):hi] = True; return m


def t0_threshold(score, valid, budget):
    """lowest quantile-grid threshold (descending probability grid, lowering the threshold step by step) with episodes/day <= budget."""
    s = score[valid]; best = None
    for q in sorted(QGRID, reverse=True):
        thr = np.quantile(s, q)
        if eps_per_day(score > thr, valid) <= budget: best = thr
        else: break
    return best if best is not None else np.quantile(s, max(QGRID))


def rolling_thr(score, q):
    return pd.Series(score).rolling(WIN7, min_periods=MINP).quantile(q).shift(1).to_numpy()


def t1_q(score, valid, budget):
    """smallest quantile level q (lowest rolling threshold) with episodes/day <= budget on the genuine DEV bars."""
    for q in QGRID:
        thr = rolling_thr(score, q); alarm = (score > np.where(np.isnan(thr), np.inf, thr))
        if eps_per_day(alarm, valid) <= budget: return q
    return QGRID[-1]


def alarms(score, rule, par):
    if rule == 'T0': return score > par
    thr = rolling_thr(score, par); return score > np.where(np.isnan(thr), np.inf, thr)


def recall_by_family(alarm, ev, seg_ok=None):
    """ev rows: t0, dur, family, (replica); detected if an alarm falls in [t0, t0+dur-1]."""
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); out = {}
    for fam, sub in ev.groupby('family'):
        hit = [(c[r.t0 + r.dur] - c[r.t0]) > 0 for r in sub.itertuples()]; out[int(fam)] = (int(np.sum(hit)), int(len(hit)))
    return out


def bootstrap_ci(hits, n=1000, seed=0):
    rng = np.random.default_rng(seed); h = np.asarray(hits, float)
    if len(h) == 0: return (np.nan, np.nan)
    b = [h[rng.integers(0, len(h), len(h))].mean() for _ in range(n)]; return (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))


def macro(rec, fams):
    v = [rec[f][0] / rec[f][1] for f in fams if f in rec and rec[f][1] > 0]; return float(np.mean(v)) if v else float('nan')


def sha_bytes(b): return hashlib.sha256(b).hexdigest()

''')

import sys, importlib; sys.path.insert(0, '.'); import b2_lib as B; importlib.reload(B); (OUT/'b2_lib.py').write_text(Path('b2_lib.py').read_text())
say('library sha256 ' + sha_file('b2_lib.py'))
assert _old_lib is None or _old_lib.strip() == Path('b2_lib.py').read_text().strip(), 'b2_lib.py differs from the copy written by notebook 28'

library sha256 1b0b46630763c944


## 2 · Data (raw bars and Layer 3 sets)

In [ ]:
def find1(pat):
    c = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c, pat; return sorted(c)[0]
FEATS = B.FEATS; AOLEN = 31 * 1440; PRIM = B.PRIMARY; BUD = B.BUDGETS; WARM = B.WARM; OLD, NEW = [1, 2, 3, 4, 5, 6], [7, 8]; ALLF = OLD + NEW
DF1 = json.load(open(OUT/'declared_final_set_v1.json')); XD = json.load(open(OUT/'xgb_v2_decisions.json')); MODE = DF1['models']['XGBoost v2']['normalisation']
CFG = XD['selected'][MODE]; FS_, SCOPE, DEPTH = CFG.split('|'); DEPTH = int(DEPTH[1:]); assert FS_ == 'A', 'v3-lite assumes the declared XGBoost v2 used feature set A'
say(f'declared XGBoost v2: normalisation {MODE}, configuration {CFG} -> v3-lite uses set C with scope {SCOPE}, depth {DEPTH}')
COLS = ['volume', 'trades_count'] + FEATS
RAW = {}; D = {}; N0 = {}; EV = {}; LAB = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a
    b = pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet', columns=COLS).iloc[:AOLEN]      # August only: TEST-B is never used
    for d_ in (a, b): d_.loc[d_.index[:15], 'volume_acceleration'] = 0.0
    RAW[(s, 'GEN_L1')] = a[COLS].reset_index(drop=True); RAW[(s, 'GEN_AO')] = b[COLS].reset_index(drop=True); N0[s] = B.fit_n0(RAW[(s, 'GEN_L1')][FEATS].iloc[15:90720])
    for name in ['TRAIN', 'VAL', 'CAL']:
        f = pd.read_parquet(L3/f'l3v3_{s}_{name}.parquet');
        if name != 'TRAIN': EV[(s, name)] = pd.read_csv(L3/f'l3v3_{s}_{name}_events.csv')
        for r, g in f.groupby('replica'):
            g = g.reset_index(drop=True); g = g.iloc[:AOLEN] if name == 'CAL' else g; RAW[(s, f'{name}_r{int(r)}')] = g[COLS]; LAB[(s, name, int(r))] = (g.label.to_numpy().astype(int), g.in_seg.to_numpy().astype(bool))
say('loaded raw bars (volume, trade count, six features) for ' + str(len(RAW)) + ' series')

declared XGBoost v2: normalisation N0, configuration A|pooled|d3 -> v3-lite uses set C with scope pooled, depth 3
loaded raw bars (volume, trade count, six features) for 14 series


## 3 · Feature set C

In [ ]:
def build_feats(Z):
    """53 causal rolling features of the six normalised features (identical recipe to XGBoost v1, look-back <= 15 bars)."""
    df = pd.DataFrame(np.asarray(Z, np.float64), columns=FEATS); cols = {}; prev15 = df.shift(1).rolling(15).mean()
    for f in FEATS:
        s_ = df[f]; a_ = s_.abs()
        cols[f + '__z'] = s_; cols[f + '__d1'] = s_ - s_.shift(1); cols[f + '__mean3'] = s_.rolling(3).mean(); cols[f + '__mean15'] = s_.rolling(15).mean(); cols[f + '__amax3'] = a_.rolling(3).max()
        cols[f + '__amax15'] = a_.rolling(15).max(); cols[f + '__std15'] = s_.rolling(15).std(); cols[f + '__dev'] = s_ - prev15[f]
    A = df.abs(); ne = (A > 3).sum(axis=1).astype(float); sa = A.sum(axis=1)
    cols['x__n_extreme'] = ne; cols['x__sumabs'] = sa; cols['x__n_extreme_mean5'] = ne.rolling(5).mean(); cols['x__sumabs_mean5'] = sa.rolling(5).mean(); cols['x__sumabs_max15'] = sa.rolling(15).max()
    out = pd.DataFrame(cols).to_numpy(np.float32, copy=True); out[:B.WARM] = np.nan; return out, list(cols)

FA_NAMES = build_feats(np.zeros((300, 6)))[1]
def build3(F, z):
    """~46 multi-scale causal features (windows end at bar t; baselines use bars strictly before t)."""
    df = pd.DataFrame(np.asarray(z, np.float64), columns=FEATS); cols = {}
    for f in FEATS:
        s_ = df[f]; cols[f + '__m60'] = s_.rolling(60).mean(); cols[f + '__m240'] = s_.rolling(240).mean(); cols[f + '__amax60'] = s_.abs().rolling(60).max(); cols[f + '__sd60'] = s_.rolling(60).std(); cols[f + '__dev240'] = s_ - s_.shift(1).rolling(240).mean()
    tf = df['trade_flow_imbalance']; lr = df['log_returns']
    for w in (15, 60, 240): cols[f'tfi__sum{w}'] = tf.rolling(w).sum()
    for w in (60, 240): cols[f'ret__sum{w}'] = lr.rolling(w).sum()
    vol = F['volume'].to_numpy(float); tr = np.maximum(F['trades_count'].to_numpy(float), 1.0)
    for nm, sr in (('vol', pd.Series(np.log1p(vol))), ('trd', pd.Series(np.log1p(tr))), ('size', pd.Series(np.log1p(vol / tr)))):
        cols[nm + '__r5_240'] = sr.rolling(5).mean() - sr.shift(5).rolling(240).mean(); cols[nm + '__r1_60'] = sr - sr.shift(1).rolling(60).mean()
    lg = pd.Series(F['large_trade_count'].to_numpy(float) / tr); cols['lgshare__dev240'] = lg - lg.shift(1).rolling(240).mean(); cols['lgshare__m15'] = lg.rolling(15).mean()
    ext = (df.abs() > 2).sum(axis=1).astype(float); cols['x__ext_m60'] = ext.rolling(60).mean(); cols['x__ext_max15'] = ext.rolling(15).max()
    sg = np.sign(tf.where(tf.abs() > 0.5, 0.0)).to_numpy(); grp = (pd.Series(sg) != pd.Series(sg).shift()).cumsum(); cols['tfi__run'] = (pd.Series(sg).groupby(grp).cumcount() + 1) * (sg != 0) * np.where(sg > 0, 1, -1)
    out = pd.DataFrame(cols).to_numpy(np.float32, copy=True); out[:WARM] = np.nan; return out, list(cols)
F3_NAMES = build3(pd.DataFrame({'volume': np.ones(400), 'trades_count': np.ones(400), 'large_trade_count': np.zeros(400)}), np.zeros((400, 6)))[1]; NAMES_C = FA_NAMES + F3_NAMES
say(f'feature set C: {len(FA_NAMES)} v2 features + {len(F3_NAMES)} multi-scale features = {len(NAMES_C)}')
XC = {}
def XOF(s, key):
    k = (s, key)
    if k not in XC:
        F = RAW[(s, key)]; z = B.normalise(F[FEATS], MODE, N0[s]); XC[k] = np.hstack([build_feats(z)[0], build3(F, z)[0]]).astype(np.float32); XC[k][:WARM] = np.nan
    return XC[k]
# fit rows (same definition as XGBoost v2: clean = no non-clean bar in [t-15, t])
LOOK = 15; RS = {}
for (s, name, r), (lab, inseg) in LAB.items():
    if name == 'CAL': continue
    n = len(lab); idx = np.arange(n); c = np.concatenate([[0], np.cumsum((lab != 0).astype(np.int64))]); clean = ((c[idx + 1] - c[np.maximum(idx - LOOK, 0)]) == 0) & inseg & (idx >= WARM); pos = (lab > 0) & inseg & (idx >= WARM); RS[(s, name, r)] = (lab, inseg, clean, pos)

feature set C: 53 v2 features + 46 multi-scale features = 99


## 4 · Unit tests

In [ ]:
s0 = SYMBOLS[0]; z = B.normalise(RAW[(s0, 'VAL_r0')][FEATS], MODE, N0[s0]); F = RAW[(s0, 'VAL_r0')]
# T01 causal: bars after t0 changed -> features before t0 unchanged
t0 = 95000; F2 = F.copy(); F2.loc[t0:, ['volume', 'trades_count']] = F2.loc[t0:, ['volume', 'trades_count']] * 7; z2 = z.copy(); z2[t0:] += 3
a1 = np.hstack([build_feats(z)[0], build3(F, z)[0]])[:t0]; a2 = np.hstack([build_feats(z2)[0], build3(F2, z2)[0]])[:t0]; assert np.allclose(a1, a2, equal_nan=True); say('PASS T01 all set-C features are causal (changing bars from t0 on leaves features before t0 unchanged)')
# T02 set C starts with the v2 set A columns (identical)
assert NAMES_C[:len(FA_NAMES)] == FA_NAMES and np.allclose(XOF(s0, 'VAL_r0')[95000:95050, :len(FA_NAMES)], build_feats(z)[0][95000:95050], equal_nan=True); say('PASS T02 set C = the 53 v2 columns + new columns')
# T03 run-length feature by hand on a toy sequence
tz = np.zeros((12, 6)); tz[:, 1] = [1, 1, 1, -1, -1, 0, 1, 1, 0.2, 1, 1, 1]; Ft = pd.DataFrame({'volume': np.ones(12), 'trades_count': np.ones(12), 'large_trade_count': np.zeros(12)}); out, nm = build3(Ft, tz); run = out[:, nm.index('tfi__run')]
sg = np.sign(pd.Series(tz[:, 1]).where(pd.Series(tz[:, 1]).abs() > 0.5, 0.0)).to_numpy(); exp = []; cur = 0; last = 0
for v in sg:
    if v == 0: cur = 0; last = 0; exp.append(0)
    elif v == last: cur += 1; exp.append(cur * int(v))
    else: cur = 1; last = v; exp.append(int(v))
chk = (pd.Series(sg).groupby((pd.Series(sg) != pd.Series(sg).shift()).cumsum()).cumcount() + 1) * (sg != 0) * np.where(sg > 0, 1, -1); assert list(chk.astype(int)) == exp, (list(chk), exp); say('PASS T03 buy/sell run length equals a hand count')
# T04 fit rows
for (s, name, r), (lab, inseg, clean, pos) in RS.items(): assert not (clean & pos).any()
assert all(RS[(s, 'VAL', r)][3].sum() > 0 for s in SYMBOLS for r in [0]); say('PASS T04 positive and clean-negative row masks are disjoint')

PASS T01 all set-C features are causal (changing bars from t0 on leaves features before t0 unchanged)
PASS T02 set C = the 53 v2 columns + new columns
PASS T03 buy/sell run length equals a hand count
PASS T04 positive and clean-negative row masks are disjoint


## 5 · Train XGBoost v3-lite (resumable)

In [ ]:
PARAMS = {'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'tree_method': 'hist', 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 5, 'reg_lambda': 1.0, 'scale_pos_weight': 5.0, 'nthread': NTHREAD}
MAX_ROUNDS, ES_ROUNDS = (25 if MOCK else 800), (8 if MOCK else 40); ROWLOG = []
def get_rows(s, name):
    Xs, ys = [], []
    for r in sorted(k[2] for k in RS if k[0] == s and k[1] == name):
        lab, inseg, clean, pos = RS[(s, name, r)]; p = pos.copy()
        if name == 'VAL': p &= (lab <= 6)
        m = p | (clean if r == 0 else np.zeros(len(lab), bool)); ii = np.flatnonzero(m); ROWLOG.append((s, name, int(ii.min()), int(ii.max()), [int(v) for v in sorted(set(lab[ii][p[ii]].tolist()))]))
        Xs.append(XOF(s, f'{name}_r{r}')[ii]); ys.append(p[ii].astype(np.float32))
    return np.concatenate(Xs), np.concatenate(ys)
def bpath(seed): return X3/f'{MODE}_C_{SCOPE}_d{DEPTH}_s{seed}.json'
def train_seed(seed):
    key_list = [('pool', SYMBOLS)] if SCOPE == 'pooled' else [(s, [s]) for s in SYMBOLS]
    for key, syms in key_list:
        p = X3/f'{MODE}_C_{SCOPE}_{key}_d{DEPTH}_s{seed}.json'; ip = Path(str(p)[:-5] + '.info.json')
        if p.exists() and ip.exists(): continue
        parts = [get_rows(s, 'TRAIN') for s in syms]; ves = [get_rows(s, 'VAL') for s in syms]
        Xt = np.concatenate([x[0] for x in parts]); yt = np.concatenate([x[1] for x in parts]); Xv = np.concatenate([x[0] for x in ves]); yv = np.concatenate([x[1] for x in ves]); t = time.time()
        bst = xgb.train({**PARAMS, 'max_depth': DEPTH, 'seed': seed}, xgb.DMatrix(Xt, label=yt, feature_names=NAMES_C), num_boost_round=MAX_ROUNDS, evals=[(xgb.DMatrix(Xv, label=yv, feature_names=NAMES_C), 'val')], early_stopping_rounds=ES_ROUNDS, verbose_eval=False)
        bst.save_model(str(p)); info = {'best_iteration': int(bst.best_iteration), 'best_aucpr_val': float(bst.best_score), 'n_fit': int(len(yt)), 'n_fit_pos': int(yt.sum()), 'n_es': int(len(yv)), 'seconds': round(time.time() - t, 1)}; ip.write_text(json.dumps(info)); say(f'  trained {p.stem}: best iteration {info["best_iteration"]}, fit {len(yt):,} rows ({int(yt.sum()):,} positive), val PR-AUC {info["best_aucpr_val"]:.3f}, {info["seconds"]}s')
TRAIN_DONE = True; say('--- training XGBoost v3-lite (set C), seeds ' + str(SEEDS) + ' ---')
for sd in SEEDS:
    if out_of_time(): TRAIN_DONE = False; break
    train_seed(sd)
if not TRAIN_DONE: say('>>> time budget reached: Run all again to continue (finished models are kept). The rest is skipped until training is done. <<<')
else:
    ok = all(r[2] >= WARM and (r[1] != 'TRAIN' or r[3] < 90720) and (r[1] != 'VAL' or (r[2] >= 90720 and r[3] < 129600)) and not any(l > 6 for l in r[4]) for r in ROWLOG); say(('PASS' if ok else 'FAIL') + ' T05 fit rows inside TRAIN-3, early-stopping rows inside VAL-3, new families never used'); assert ok

--- training XGBoost v3-lite (set C), seeds [11, 22, 33] ---
  trained N0_C_pooled_pool_d3_s11: best iteration 799, fit 175,506 rows (23,870 positive), val PR-AUC 0.791, 40.3s
  trained N0_C_pooled_pool_d3_s22: best iteration 789, fit 175,506 rows (23,870 positive), val PR-AUC 0.792, 37.7s
  trained N0_C_pooled_pool_d3_s33: best iteration 796, fit 175,506 rows (23,870 positive), val PR-AUC 0.791, 37.4s
PASS T05 fit rows inside TRAIN-3, early-stopping rows inside VAL-3, new families never used


## 6 · Scores

In [ ]:
class V3:
    def __init__(s_): s_.b = {}
    def boosters(s_, sym):
        key = 'pool' if SCOPE == 'pooled' else sym
        if key not in s_.b:
            out = []
            for sd in SEEDS:
                p = X3/f'{MODE}_C_{SCOPE}_{key}_d{DEPTH}_s{sd}.json'; b_ = xgb.Booster(); b_.load_model(str(p)); out.append((b_, json.loads(Path(str(p)[:-5] + '.info.json').read_text())['best_iteration']))
            s_.b[key] = out
        return s_.b[key]
    def score(s_, sym, key):
        dm = xgb.DMatrix(XOF(sym, key), feature_names=NAMES_C); m = np.mean([b_.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for b_, bi in s_.boosters(sym)], axis=0).astype('float64'); m[:WARM] = np.nan; return m
M3 = V3() if TRAIN_DONE else None
S3 = {}; S2 = {}
if TRAIN_DONE:
    for s in SYMBOLS:
        z = np.load(OUT/'scores'/f'xgb_v2_{s}_{MODE}.npz'); d2 = {}; d3 = {}
        for k in z.files:
            if k.startswith('TRAIN'): continue
            x = z[k].astype('float64'); d2[k] = x[:AOLEN] if k.startswith('CAL') else x
        for k in d2: d3[k] = M3.score(s, k)
        S2[s] = d2; S3[s] = d3
    say('scored all genuine, VAL-3 and CAL-3 series with v3-lite; TRAIN series are not scored (no test of in-sample behaviour needed)')

scored all genuine, VAL-3 and CAL-3 series with v3-lite; TRAIN series are not scored (no test of in-sample behaviour needed)


## 7 · VAL-3: v2 versus v3-lite

In [ ]:
L1V = B.valid_mask(129600, 90720, 129600); CALV = B.valid_mask(AOLEN, 1440, AOLEN)
def thr_roll(g, q): t = B.rolling_thr(g, q); return np.where(np.isnan(t), np.inf, t)
def event_hits(a, ev):
    c = np.concatenate([[0], np.cumsum(np.nan_to_num(a).astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
def sets_of(name):
    out = []
    for s in SYMBOLS:
        ev = EV[(s, name)]
        for r in sorted(ev.replica.unique()): out.append((s, f'{name}_r{int(r)}', ev[ev.replica == r].reset_index(drop=True)))
    return out
ES_ = {n: sets_of(n) for n in ['VAL', 'CAL']}; META = {n: pd.concat([pd.DataFrame({'sym': s, 'fam': ev.family.to_numpy()}) for s, k, ev in ES_[n]], ignore_index=True) for n in ES_}
def run(S, name, bud):
    """T1c: q from the genuine DEV series, rolling threshold from the genuine reference (VAL: GEN_L1, CAL: GEN_AO) applied to the injected series. Returns event hits (all events) and genuine DEV / CAL episodes per day."""
    ref = 'GEN_L1' if name == 'VAL' else 'GEN_AO'; hs = []; dv = []; cv = []
    for s in SYMBOLS:
        q = B.t1_q(S[s]['GEN_L1'], L1V, bud); thr = thr_roll(S[s][ref], q); dv.append(B.eps_per_day(S[s]['GEN_L1'] > thr_roll(S[s]['GEN_L1'], q), L1V)); cv.append(B.eps_per_day(S[s]['GEN_AO'] > thr_roll(S[s]['GEN_AO'], q), CALV))
        for s2, key, ev in ES_[name]:
            if s2 == s: hs.append(event_hits(S[s][key] > thr, ev))
    return np.concatenate(hs), float(np.mean(dv)), float(np.mean(cv))
def groups(name, fams):
    m = META[name]; G = []
    for s in SYMBOLS:
        G.append([np.flatnonzero(((m.sym == s) & (m.fam == f)).to_numpy()) for f in fams if ((m.sym == s) & (m.fam == f)).any()])
    return G
def macro(h, name, fams): return float(np.mean([np.mean([h[i].mean() for i in gs]) for gs in groups(name, fams)]))
def fam_rates(h, name): m = META[name]; return pd.DataFrame({'sym': m.sym, 'fam': m.fam, 'h': h}).groupby(['sym', 'fam']).h.mean().groupby('fam').mean()
def paired(h1, h2, name, fams, n=None, seed=0):
    n = n or NBOOT; G = groups(name, fams); rg = np.random.default_rng(seed); out = np.zeros(n)
    for i in range(n):
        acc = []
        for gs in G:
            acc.append(np.mean([h1[idx[p]].mean() - h2[idx[p]].mean() for idx in gs for p in [rg.integers(0, len(idx), len(idx))]]))
        out[i] = np.mean(acc)
    return out
def paired_fam(h1, h2, name, fam, n=None, seed=0):
    n = n or NBOOT; m = META[name]; rg = np.random.default_rng(seed); idxs = [np.flatnonzero(((m.sym == s) & (m.fam == fam)).to_numpy()) for s in SYMBOLS]; out = np.zeros(n)
    for i in range(n): out[i] = np.mean([(h1[idx[p]].mean() - h2[idx[p]].mean()) for idx in idxs for p in [rg.integers(0, len(idx), len(idx))]])
    return out
def ci(x): return f'{np.percentile(x, 2.5):+.2f}..{np.percentile(x, 97.5):+.2f}'
RES = {}
if TRAIN_DONE:
    rows = []
    for nm, S in [('XGBoost v2', S2), ('XGBoost v3-lite', S3)]:
        for bud in BUD:
            h, dv, cv = run(S, 'VAL', bud); RES[(nm, 'VAL', bud)] = h
            rows.append({'model': nm, 'budget': bud, 'DEV genuine eps/day': dv, 'recall 6 old': macro(h, 'VAL', OLD), 'recall 2 new': macro(h, 'VAL', NEW), 'macro 8': macro(h, 'VAL', ALLF)})
    VT = pd.DataFrame(rows); say('--- VAL-3 (T1c): XGBoost v2 versus v3-lite ---'); say(VT.round(3).to_string(index=False))
    h2, h3 = RES[('XGBoost v2', 'VAL', PRIM)], RES[('XGBoost v3-lite', 'VAL', PRIM)]; r2, r3 = fam_rates(h2, 'VAL'), fam_rates(h3, 'VAL')
    FT = pd.DataFrame({'v2': r2, 'v3-lite': r3, 'difference': r3 - r2}); FT['diff 95% CI'] = [ci(paired_fam(h3, h2, 'VAL', f)) for f in FT.index]; FT.index = [B.FAMS[i] for i in FT.index]
    say(''); say('--- VAL-3 per family, budget 2/day (paired event bootstrap of the difference) ---'); say(FT.round(3).to_string())

--- VAL-3 (T1c): XGBoost v2 versus v3-lite ---
          model  budget  DEV genuine eps/day  recall 6 old  recall 2 new  macro 8
     XGBoost v2       1                0.815         0.662         0.750    0.684
     XGBoost v2       2                1.500         0.695         0.792    0.719
     XGBoost v2       4                2.963         0.724         0.833    0.751
XGBoost v3-lite       1                0.833         0.647         0.729    0.667
XGBoost v3-lite       2                1.704         0.689         0.785    0.713
XGBoost v3-lite       4                3.074         0.717         0.854    0.751

--- VAL-3 per family, budget 2/day (paired event bootstrap of the difference) ---
                              v2  v3-lite  difference   diff 95% CI
pump_and_dump_like         0.724    0.737       0.013  -0.07..+0.08
fake_breakout_like         0.618    0.618       0.000  -0.07..+0.07
abnormal_volume_burst      0.092    0.132       0.039  -0.03..+0.12
trade_flow_imbalance    

## 8 · Acceptance rule (amendment A4)

In [ ]:
DEC = {}
if TRAIN_DONE:
    d8 = paired(h3, h2, 'VAL', ALLF); diff8 = macro(h3, 'VAL', ALLF) - macro(h2, 'VAL', ALLF); a_ = bool(diff8 >= 0.03 and np.percentile(d8, 2.5) > 0)
    drops = (r3 - r2); b_ = bool((drops >= -0.05).all()); _, dev3, _ = run(S3, 'VAL', PRIM); c_ = bool(0.5 * PRIM <= dev3 <= 2 * PRIM); ACCEPT = a_ and b_ and c_
    verdict36 = (OUT/'diag36_verdict.csv'); flagged = []
    if verdict36.exists():
        vt = pd.read_csv(verdict36); flagged = [int(k) for k, nm_ in B.FAMS.items() if nm_ in set(vt[vt.verdict.str.startswith('FEATURE')].family)]
    hv3 = {B.FAMS[f]: {'difference': float(r3[f] - r2[f]), 'holds (>= 0.05)': bool(r3[f] - r2[f] >= 0.05)} for f in flagged}
    DEC = {'macro 8 difference': diff8, 'macro 8 difference 95% CI': ci(d8), '(a) >= 0.03 and CI above 0': a_, '(b) no family drops by more than 0.05': b_, '(c) DEV genuine eps/day within [0.5,2] x budget': c_, 'DEV genuine eps/day (T1)': dev3,
           'ACCEPT v3-lite as the XGBoost of the final set': bool(ACCEPT), 'families flagged by notebook 36 (feature/model gap)': [B.FAMS[f] for f in flagged] if verdict36.exists() else 'notebook 36 output not found (run it for the family-level H-V3)', 'H-V3 per flagged family': hv3}
    say(''); say('--- acceptance rule (amendment A4) ---'); [say(f'{k}: {v}') for k, v in DEC.items()]
    say('RESULT: ' + ('v3-lite is ACCEPTED and replaces XGBoost v2 in the final set (declared_final_set_v2.json is written; notebook 32 and fusion must be re-run on it before TEST).' if ACCEPT else 'v3-lite is NOT accepted: XGBoost v2 stays in the final set; v3-lite is reported as a negative result (the per-family table above shows what changed).'))


--- acceptance rule (amendment A4) ---
macro 8 difference: -0.006670321637426868
macro 8 difference 95% CI: -0.03..+0.01
(a) >= 0.03 and CI above 0: False
(b) no family drops by more than 0.05: False
(c) DEV genuine eps/day within [0.5,2] x budget: True
DEV genuine eps/day (T1): 1.7037037037037037
ACCEPT v3-lite as the XGBoost of the final set: False
families flagged by notebook 36 (feature/model gap): ['abnormal_volume_burst']
H-V3 per flagged family: {'abnormal_volume_burst': {'difference': 0.039473684210526314, 'holds (>= 0.05)': False}}
RESULT: v3-lite is NOT accepted: XGBoost v2 stays in the final set; v3-lite is reported as a negative result (the per-family table above shows what changed).


## 9 · CAL-3 (report once)

In [ ]:
if TRAIN_DONE:
    rows = []
    for nm, S in [('XGBoost v2', S2), ('XGBoost v3-lite', S3)]:
        for bud in BUD:
            h, dv, cv = run(S, 'CAL', bud); RES[(nm, 'CAL', bud)] = h
            rows.append({'model': nm, 'budget': bud, 'CAL genuine eps/day': cv, 'recall 6 old': macro(h, 'CAL', OLD), 'recall 2 new': macro(h, 'CAL', NEW), 'macro 8': macro(h, 'CAL', ALLF)})
    CT = pd.DataFrame(rows); say(''); say('--- CAL-3 (reported once, not used for the decision): v2 versus v3-lite, T1c ---'); say(CT.round(3).to_string(index=False))
    c2, c3 = fam_rates(RES[('XGBoost v2', 'CAL', PRIM)], 'CAL'), fam_rates(RES[('XGBoost v3-lite', 'CAL', PRIM)], 'CAL'); CF = pd.DataFrame({'v2': c2, 'v3-lite': c3, 'difference': c3 - c2}); CF.index = [B.FAMS[i] for i in CF.index]; say(''); say('--- CAL-3 per family, budget 2/day ---'); say(CF.round(3).to_string())


--- CAL-3 (reported once, not used for the decision): v2 versus v3-lite, T1c ---
          model  budget  CAL genuine eps/day  recall 6 old  recall 2 new  macro 8
     XGBoost v2       1                0.783         0.690          0.72    0.698
     XGBoost v2       2                1.167         0.710          0.75    0.720
     XGBoost v2       4                2.500         0.750          0.81    0.765
XGBoost v3-lite       1                0.900         0.660          0.71    0.673
XGBoost v3-lite       2                1.733         0.693          0.75    0.708
XGBoost v3-lite       4                2.900         0.737          0.78    0.748

--- CAL-3 per family, budget 2/day ---
                             v2  v3-lite  difference
pump_and_dump_like         0.68     0.68       -0.00
fake_breakout_like         0.82     0.78       -0.04
abnormal_volume_burst      0.10     0.10        0.00
trade_flow_imbalance       0.84     0.80       -0.04
wash_like_aggregate_proxy  1.00     0.9

## 10 · Save

In [ ]:
if TRAIN_DONE:
    VT.to_csv(OUT/'xgb_v3_val_table.csv', index=False); FT.to_csv(OUT/'xgb_v3_val_by_family.csv'); CT.to_csv(OUT/'xgb_v3_cal_table.csv', index=False); CF.to_csv(OUT/'xgb_v3_cal_by_family.csv')
    for s in SYMBOLS: np.savez_compressed(OUT/'scores'/f'xgb_v3_{s}_{MODE}.npz', **{k: v.astype(np.float32) for k, v in S3[s].items()})
    DEC['amendment_A4_sha256'] = A4H; DEC['config'] = {'normalisation': MODE, 'scope': SCOPE, 'depth': DEPTH, 'features': len(NAMES_C), 'seeds': SEEDS}; (OUT/'xgb_v3_decisions.json').write_text(json.dumps(DEC, indent=1, default=str))
    if DEC['ACCEPT v3-lite as the XGBoost of the final set']:
        d2 = json.loads((OUT/'declared_final_set_v1.json').read_text()); d2['status'] = 'RE-DECLARED (v2): XGBoost v3-lite accepted by amendment A4 before TEST was opened. Supersedes declared_final_set_v1.json for XGBoost only; q values must be recomputed by the notebook that uses it.'
        d2['models']['XGBoost v2'] = {'normalisation': MODE, 'configuration': f'v3-lite set C|{SCOPE}|d{DEPTH}', 'seeds': SEEDS, 'score_file_sha256_16': {s: [f'scores/xgb_v3_{s}_{MODE}.npz', sha_file(OUT/'scores'/f'xgb_v3_{s}_{MODE}.npz')] for s in SYMBOLS}, 'note': 'key name kept for compatibility; scores come from xgb_v3'}
        d2['amendment_A4_sha256'] = A4H; (OUT/'declared_final_set_v2.json').write_text(json.dumps(d2, indent=1)); say('wrote declared_final_set_v2.json')
    say('saved xgb_v3 tables, scores and xgb_v3_decisions.json')
block = '\n'.join(LINES); (OUT/'b2_step8_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step8_share_block.txt, or copy below) =====\n' + block[-12000:])

saved xgb_v3 tables, scores and xgb_v3_decisions.json

===== SHARE BLOCK (attach models_b2_v1/b2_step8_share_block.txt, or copy below) =====
protocol B2 intact, sha256 443f99d5faf5929f; amendment A4 sha256 3986409d2b6aacd8; xgboost 3.4.1; 2 threads
library sha256 1b0b46630763c944
declared XGBoost v2: normalisation N0, configuration A|pooled|d3 -> v3-lite uses set C with scope pooled, depth 3
loaded raw bars (volume, trade count, six features) for 14 series
feature set C: 53 v2 features + 46 multi-scale features = 99
PASS T01 all set-C features are causal (changing bars from t0 on leaves features before t0 unchanged)
PASS T02 set C = the 53 v2 columns + new columns
PASS T03 buy/sell run length equals a hand count
PASS T04 positive and clean-negative row masks are disjoint
--- training XGBoost v3-lite (set C), seeds [11, 22, 33] ---
  trained N0_C_pooled_pool_d3_s11: best iteration 799, fit 175,506 rows (23,870 positive), val PR-AUC 0.791, 40.3s
  trained N0_C_pooled_pool_d3_s22: best it